# Inteligência Artificial e Aprendizado de Máquina — Entrega 1
## ASA-Connect: Agente de Pendências

**Objetivo:** criar uma primeira versão do agente para ajudar no atendimento de dúvidas sobre financeiro, rematrícula, bolsas e documentos. Uma Árvore de Decisão identifica o assunto do pedido, e regras simples mostram o que o atendente precisa conferir, uma prioridade inicial e o setor indicado. A decisão final continua sendo do atendente.

O código, o relatório técnico e a primeira versão do Model Card estão reunidos neste notebook.


## 1. Análise dos dados

Recebemos cinco planilhas e uma base já unificada. A tabela mostra o tamanho de cada arquivo **antes de qualquer limpeza**, o que encontramos nele e o que cada linha representa.

| Fonte | Registros | O que contém | O que representa cada linha |
|---|---:|---|---|
| `Contato.xlsx` | 17.812 | Dados cadastrais, como cidade, bairro e idade | Cadastro de um aluno; há alguns IDs repetidos |
| `Financeiro.xlsx` | 538.044 | Lançamentos, parcelas, valores, bolsas e datas de vencimento/pagamento | Um lançamento financeiro; um aluno pode aparecer várias vezes |
| `Historico.xlsx` | 274.901 | Disciplinas, notas, frequência e situação acadêmica | Registro de disciplina/período do aluno |
| `Matriculas.xlsx` | 20.920 | Curso, turno, período e situação acadêmica | Registro de matrícula/período; pode haver mais de um por aluno |
| `Relacionamentos.xlsx` | 96.628 | Tema, motivo e data dos contatos feitos | Um registro de relacionamento/atendimento |
| `Base Unificada.csv` | 12.842 | Indicadores reunidos das fontes anteriores | Resumo por aluno, com 24 colunas; há 7 IDs repetidos |

*As contagens foram conferidas nos arquivos recebidos e não representam, necessariamente, a mesma quantidade de alunos em todas as fontes.*

Usamos a `Base Unificada.csv` como ponto de partida porque ela já reúne indicadores financeiros, acadêmicos e de atendimento por `ID_ALUNO`. Assim, conseguimos fazer a demonstração sem juntar todas as planilhas novamente a cada execução. A base reúne **12.835 IDs distintos**; as repetições são tratadas na consulta demonstrativa, na próxima seção.

Os indicadores ajudam a perceber situações que merecem conferência, mas não mostram tudo. Por exemplo: ter parcelas em aberto é um sinal para consultar o financeiro, não uma confirmação de bloqueio da rematrícula. A base também não informa quais documentos estão faltando nem todas as regras ou prazos do ASA.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# No Colab, envie o CSV quando for solicitado.
# No computador, basta deixar o CSV na mesma pasta do notebook.
if not Path("Base Unificada.csv").exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError("Coloque 'Base Unificada.csv' na mesma pasta do notebook.")

df = pd.read_csv("Base Unificada.csv")
print("Registros:", len(df), "| Colunas:", len(df.columns))
print("Alunos distintos:", df["ID_ALUNO"].nunique())

# Exibimos só informações gerais, sem divulgar registros individuais.
colunas_resumo = ["pct_parcelas_em_aberto", "pct_parcelas_acordo",
                  "atraso_medio_dias", "total_contatos"]
display(df[colunas_resumo].describe().round(2))


## 2. Preparação dos dados

As cinco planilhas têm `ID_ALUNO` em comum, mas algumas possuem várias linhas para o mesmo estudante. Para criar uma base por aluno, seria necessário agrupar lançamentos, disciplinas e contatos antes de juntar as informações. *
Para nossa consulta, verificamos IDs vazios e repetidos, mantivemos uma linha por estudante e convertemos os indicadores numéricos para o formato correto. Se algum número for inválido, ele vira um valor ausente — e não um sinal de que não existe pendência. Como a base não indica qual registro repetido é o mais recente, manter a primeira ocorrência é apenas uma escolha provisória para a demonstração.

Também observamos que `atraso_medio_dias` pode ser negativo. Sem a definição desse indicador, **não tratamos valores negativos como atraso confirmado** nem os usamos para aumentar ou reduzir a prioridade. Não usamos idade, sexo ou bairro para definir prioridades.


In [ ]:
print("IDs vazios:", df["ID_ALUNO"].isna().sum())
print("IDs repetidos:", df["ID_ALUNO"].duplicated().sum())
print("Campos vazios por coluna (os 8 maiores):")
display(df.isna().sum().sort_values(ascending=False).head(8).to_frame("Quantidade"))

dados = df.dropna(subset=["ID_ALUNO"]).drop_duplicates("ID_ALUNO").copy()
colunas_numericas = ["pct_parcelas_em_aberto", "pct_parcelas_acordo",
                    "atraso_medio_dias", "total_contatos", "media_assiduidade"]
for coluna in colunas_numericas:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")

print("Alunos disponíveis para consulta:", len(dados))
print("Valores de atraso negativo (precisam de conferência):",
      (dados["atraso_medio_dias"] < 0).sum())
print("Valores ausentes nos atributos escolhidos:")
display(dados[colunas_numericas].isna().sum().to_frame("Quantidade"))


## 3. Seleção de atributos (features)

Separamos as informações em dois grupos: **o texto do pedido**, usado pelo modelo para identificar o assunto, e **os dados do aluno**, consultados pelas regras para montar a orientação.

| Dimensão | Atributo | Por que foi escolhido | Uso nesta versão |
|---|---|---|---|
| Solicitação | Texto do pedido | Permite reconhecer se a dúvida é sobre financeiro, rematrícula, bolsa ou documentos | Entrada da Árvore de Decisão |
| Identificação | `ID_ALUNO` | Localiza os indicadores do estudante, sem usar seu nome | Consulta, fora do modelo |
| Financeira | `pct_parcelas_em_aberto` | Indica a proporção de parcelas em aberto (valor entre 0 e 1) | Conferência financeira |
| Financeira | `atraso_medio_dias` | Ajuda a organizar casos financeiros, desde que o valor seja válido | Prioridade financeira provisória |
| Financeira | `pct_parcelas_acordo` | Sinaliza acordos registrados que merecem conferência | Complementa o checklist financeiro |
| Bolsas | `tem_bolsa` | Mostra se há registro de bolsa na base, mas não se a renovação foi aprovada | Contexto para pedidos de bolsa |
| Acadêmica | `media_assiduidade` | Mostra a frequência média e pode ajudar em verificações futuras | Contexto; não define prioridade agora |
| Atendimento | `total_contatos` | Mostra quantos contatos anteriores foram registrados | Contexto; não define prioridade agora |

A Árvore de Decisão **não é treinada com a base de alunos**. Ela aprende o assunto de 40 frases fictícias. Os indicadores do CSV são utilizados depois, nas regras de atendimento. Não usamos `evadiu`, pois nosso agente não pretende prever evasão.


## 4. Métricas escolhidas

Nesta primeira entrega, usamos **acurácia** para ver quantas frases de teste tiveram seu assunto identificado corretamente. Também mostramos precisão, recall e F1 por assunto para observar onde o modelo mais erra, mas a amostra é pequena e fictícia. A verificação das respostas do agente será feita com quatro pedidos de exemplo, conferindo o assunto, a presença do checklist e o setor indicado.

Na próxima entrega, queremos testar solicitações reais autorizadas e revisadas. Além de acurácia, precisão, recall e F1, pretendemos medir o **acerto dos encaminhamentos** e dos checklists com ajuda de atendentes do ASA. Nesta fase, ainda não podemos afirmar que as sugestões estão corretas para casos reais.


## 5b. Baseline do Agente de Pendências

### 5b.1 Modelo inicial: Árvore de Decisão

Começamos com 40 frases fictícias, dez para cada assunto: financeiro, rematrícula, bolsas e documentos. O `CountVectorizer` transforma as palavras em números; a Árvore de Decisão usa esses números para classificar a pergunta. Separamos 75% das frases para treino e 25% para teste. Mantemos essa primeira versão simples para mostrar todo o caminho, da pergunta à orientação do atendente.


In [ ]:
exemplos = {
    "financeiro": [
        "tenho boleto atrasado", "quero verificar parcelas em aberto", "preciso pagar mensalidade",
        "tenho dívida de pagamento", "como consultar meu acordo financeiro", "meu boleto está vencido",
        "quero regularizar parcelas", "minha mensalidade está atrasada",
        "preciso consultar pagamento", "há algum débito financeiro"],
    "rematricula": [
        "quero fazer rematrícula", "como renovar minha matrícula", "preciso me rematricular",
        "qual o procedimento de rematrícula", "quero continuar matriculado", "como fazer matrícula do semestre",
        "tenho dúvidas sobre rematrícula", "preciso renovar matrícula",
        "quando posso fazer rematrícula", "como concluir a matrícula"],
    "bolsa": [
        "quero renovar minha bolsa", "tenho dúvida sobre bolsa de estudos", "como solicitar bolsa",
        "minha bolsa está ativa", "preciso consultar desconto da bolsa", "quero saber sobre auxílio estudantil",
        "minha bolsa precisa de renovação", "como verificar benefício estudantil",
        "preciso regularizar bolsa", "quero informação sobre bolsa"],
    "documentos": [
        "quais documentos faltam", "preciso entregar documentação", "tenho documento pendente",
        "como enviar comprovante", "falta algum documento meu", "quero verificar documentação",
        "preciso atualizar meus documentos", "entrega de histórico escolar",
        "meu comprovante foi recebido", "quero consultar documento"],
}
frases = [frase for lista in exemplos.values() for frase in lista]
assuntos = [assunto for assunto, lista in exemplos.items() for _ in lista]
X_treino, X_teste, y_treino, y_teste = train_test_split(
    frases, assuntos, test_size=0.25, random_state=42, stratify=assuntos
)
modelo = make_pipeline(
    CountVectorizer(strip_accents="unicode", ngram_range=(1, 2)),
    DecisionTreeClassifier(random_state=42)
)
modelo.fit(X_treino, y_treino)
previsoes = modelo.predict(X_teste)
acuracia = accuracy_score(y_teste, previsoes)
print(f"Acurácia no teste fictício: {acuracia:.0%} ({sum(previsoes == y_teste)}/{len(y_teste)} frases)")
print(classification_report(y_teste, previsoes, zero_division=0))
comparacao = pd.DataFrame({"Solicitação fictícia": X_teste, "Esperado": y_teste,
                           "Previsto": previsoes})
display(comparacao)
print("Exemplos em que o modelo errou:")
display(comparacao[comparacao["Esperado"] != comparacao["Previsto"]])


### 5b.2 Regras e tabela de decisão

Depois de classificar o assunto, o agente consulta os indicadores disponíveis e aplica regras simples de `if/else`. A tabela abaixo resume essas regras. As prioridades e o limite de **30 dias** são escolhas de demonstração, não regras oficiais do ASA. Quando a informação é incompleta, indicamos **"A verificar"** em vez de adivinhar uma pendência.


In [ ]:
tabela_decisao = pd.DataFrame([
    ["Financeiro: parcelas em aberto e atraso válido de 30 dias ou mais",
     "Conferir parcelas e possíveis acordos", "Alta (provisória)", "Financeiro"],
    ["Financeiro: parcelas em aberto e atraso válido inferior a 30 dias",
     "Conferir parcelas e possíveis acordos", "Média (provisória)", "Financeiro"],
    ["Financeiro: parcelas em aberto e atraso ausente/negativo",
     "Consultar datas e situação atual", "A verificar", "Financeiro"],
    ["Financeiro: sem parcelas em aberto ou indicador ausente",
     "Confirmar situação atual no sistema", "A verificar", "Financeiro"],
    ["Rematrícula", "Conferir calendário, documentos e requisitos", "A verificar", "Secretaria acadêmica"],
    ["Bolsa", "Conferir registro, condições e renovação", "A verificar", "Setor de bolsas"],
    ["Documentos", "Conferir exigências e recebimento", "A verificar", "Secretaria acadêmica"],
], columns=["Condição", "Resultado", "Prioridade", "Setor"])
display(tabela_decisao)


### Como definimos a prioridade inicial

Para os pedidos financeiros, usamos a presença de parcelas em aberto e o atraso médio: **30 dias ou mais = alta**, **menos de 30 dias = média**, desde que o atraso seja um número válido. Se não há dados suficientes, o resultado fica como **"A verificar"**. Para os demais assuntos, ainda precisamos dos registros e das regras oficiais antes de definir uma ordem de atendimento. Esse limite de 30 dias é apenas um exemplo para a primeira entrega.


In [ ]:
def orientar_aluno(id_aluno, pedido):
    """Classifica o assunto e devolve uma orientação inicial para o atendente."""
    aluno = dados.loc[dados["ID_ALUNO"].astype(str) == str(id_aluno)]
    if aluno.empty:
        return {"Assunto": "Não localizado", "Checklist": "Conferir identificação",
                "Prioridade": "A verificar", "Próxima ação": "Localizar cadastro",
                "Justificativa": "ID não encontrado", "Setor": "Atendimento"}

    assunto = modelo.predict([pedido])[0]
    registro = aluno.iloc[0]

    if assunto == "financeiro":
        aberto = registro["pct_parcelas_em_aberto"]
        atraso = registro["atraso_medio_dias"]
        acordo = registro["pct_parcelas_acordo"]

        if pd.isna(aberto) or not (0 <= aberto <= 1):
            checklist = "Consultar parcelas diretamente no sistema"
            prioridade = "A verificar"
            motivo = "Indicador de parcelas em aberto ausente ou inválido"
        elif aberto > 0:
            checklist = "Conferir parcelas em aberto"
            if pd.notna(acordo) and acordo > 0:
                checklist += " e verificar se os acordos registrados continuam válidos"
            if pd.isna(atraso) or atraso < 0:
                prioridade = "A verificar"
                motivo = "Há indicador de parcelas em aberto, mas o atraso precisa ser conferido"
            elif atraso >= 30:
                prioridade = "Alta (provisória)"
                motivo = "Há parcelas em aberto e o atraso médio informado é de 30 dias ou mais"
            else:
                prioridade = "Média (provisória)"
                motivo = "Há parcelas em aberto e o atraso médio informado é inferior a 30 dias"
        else:
            checklist = "Confirmar a situação financeira atual"
            prioridade = "A verificar"
            motivo = "O indicador resumido não aponta parcelas em aberto"
        return {"Assunto": assunto, "Checklist": checklist, "Prioridade": prioridade,
                "Próxima ação": "Conferir no sistema financeiro e orientar o aluno",
                "Justificativa": motivo, "Setor": "Financeiro"}

    if assunto == "bolsa":
        status = str(registro.get("tem_bolsa", "")).lower()
        situacao = ("Existe registro de bolsa na base, mas precisamos confirmar o status atual"
                    if status == "true" else
                    "A base não indica bolsa ativa; conferir se há solicitação ou benefício recente"
                    if status == "false" else
                    "A situação da bolsa não está clara na base")
        return {"Assunto": assunto,
                "Checklist": "Conferir condições, documentos e situação da bolsa",
                "Prioridade": "A verificar",
                "Próxima ação": "Consultar a situação e as regras de renovação",
                "Justificativa": situacao, "Setor": "Setor de bolsas"}

    orientacoes = {
        "rematricula": ("Conferir calendário, documentos e requisitos da rematrícula",
                        "Consultar a situação atual e as regras oficiais", "Secretaria acadêmica"),
        "documentos": ("Conferir documentos exigidos e recebimento",
                       "Verificar documentos pendentes no sistema", "Secretaria acadêmica"),
    }
    checklist, proxima, setor = orientacoes[assunto]
    return {"Assunto": assunto, "Checklist": checklist, "Prioridade": "A verificar",
            "Próxima ação": proxima,
            "Justificativa": "A base não confirma a pendência específica deste procedimento",
            "Setor": setor}


### 5b.3 Verificação inicial

Vamos fazer quatro pedidos de exemplo, um para cada assunto. O ID usado nesta demonstração é retirado da base **somente durante a execução** e não aparece nos resultados. Selecionamos um aluno com indicador de parcelas em aberto e atraso válido, quando disponível, para mostrar a regra financeira.

A tabela compara o assunto esperado com o reconhecido pelo modelo. Também conferimos se a resposta apresenta checklist e encaminhamento.


In [ ]:
# Escolhemos um registro que permita demonstrar a regra financeira.
candidatos = dados.loc[(dados["pct_parcelas_em_aberto"] > 0) &
                       (dados["atraso_medio_dias"] >= 30)]
id_exemplo = candidatos.iloc[0]["ID_ALUNO"] if not candidatos.empty else dados.iloc[0]["ID_ALUNO"]

pedidos_demo = [
    ("financeiro", "quero verificar parcelas em aberto"),
    ("rematricula", "quero fazer rematrícula"),
    ("bolsa", "quero renovar minha bolsa"),
    ("documentos", "quais documentos faltam"),
]
resultados_demo = []
for esperado, pedido in pedidos_demo:
    resposta = orientar_aluno(id_exemplo, pedido)
    print("\nSOLICITAÇÃO:", pedido)
    display(pd.DataFrame([resposta]))
    resultados_demo.append({
        "Solicitação": pedido,
        "Assunto esperado": esperado,
        "Assunto identificado": resposta["Assunto"],
        "Assunto correto?": "Sim" if resposta["Assunto"] == esperado else "Não",
        "Checklist presente?": "Sim" if resposta.get("Checklist") else "Não",
        "Setor indicado": resposta["Setor"],
    })

print("Resumo dos quatro testes demonstrativos:")
display(pd.DataFrame(resultados_demo))


## 6. Limitações conhecidas

- Treinamos o classificador com apenas **40 frases inventadas**; perguntas reais podem ser diferentes, misturar assuntos ou ser classificadas incorretamente.
- A base unificada tem IDs repetidos e muitos valores negativos em `atraso_medio_dias`. Precisamos entender melhor a origem desses valores antes de usar os dados em situações reais.
- Não temos a lista de documentos por estudante, a situação atual de cada solicitação nem as regras e os prazos completos do ASA.
- As prioridades são provisórias. A acurácia das frases fictícias não comprova que o agente faria encaminhamentos corretos no atendimento real.


## 7. Próximos passos

Na Entrega 2, queremos incluir as regras oficiais para rematrícula, bolsas e documentos e avaliar os checklists e encaminhamentos. Também será importante entender os valores negativos de atraso e conferir se a forma de agrupar as fontes está correta.


---
# Parte 2 — Model Card (primeira versão)

**Nome:** ASA-Connect — Agente de Pendências  
**Versão:** 1.0 (baseline)  
**Data e responsáveis:** identificados no histórico do repositório do grupo no GitHub  
**Modelo:** `CountVectorizer` + `DecisionTreeClassifier` para identificar o assunto. Depois, regras `if/else` e uma tabela de decisão montam a orientação.

**Uso pretendido:** apoiar atendentes do ASA na primeira conferência de solicitações sobre financeiro, rematrícula, bolsas e documentos. O agente apresenta checklist, prioridade provisória, próximo passo, justificativa e setor. Não serve para aprovar ou negar pedidos, bloquear matrículas, aplicar punições nem substituir a análise humana.

**Dados utilizados:** o classificador foi treinado com **40 frases fictícias**, sendo dez por assunto. A `Base Unificada.csv` tem **12.842 registros, 12.835 IDs distintos e 24 colunas**; ela é consultada pelas regras, **não usada para treinar o classificador de textos**. Os indicadores principais desta versão são parcelas em aberto, atraso, acordos e registro de bolsa. O período de cobertura e o processo original de unificação não foram confirmados.

**Avaliação preliminar:** na execução de referência, acertamos **6 de 10 frases de teste (60% de acurácia)**, com divisão fixa de 75% para treino e 25% para teste. Também executamos quatro solicitações de exemplo para verificar se a resposta inclui assunto, checklist e setor. Esses testes mostram o protótipo funcionando, mas não medem seu desempenho em situações reais.

**Cuidados éticos:** consultar apenas dados autorizados, não publicar registros individuais dos alunos e confirmar informações nos sistemas oficiais. As decisões e qualquer medida que afete o estudante continuam sendo responsabilidade dos atendentes.

**Limitações:** os exemplos de treino são poucos e fictícios, e o modelo pode confundir assuntos. Faltam dados documentais, regras oficiais completas e verificação dos indicadores de atraso. As prioridades ainda precisam de validação com o ASA.

**Histórico:** versão 1.0 — primeira baseline para a Entrega 1. O registro de entrega fica no GitHub.
